# RG3 가스12개 + 보완 미성형16개 전처리/학습
새 상태 미성형 검출은 아직 정답으로 검증하지 못한 개발 후보임을 README에서 확인하세요.


In [ ]:
from pathlib import Path
import os,sys,json,joblib,nbformat
import numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from scipy.special import expit
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'rg3/01_preprocess_train.ipynb').exists())
WORK=ROOT/'rg3/internal';MODEL_DIR=ROOT/'rg3/models';MODEL_DIR.mkdir(exist_ok=True)
def source(name):
 return ''.join(c.source for c in nbformat.read(WORK/'notebooks'/name,4).cells if c.cell_type=='code')
gas_source=source('154_rg3_gas_model_export.ipynb')
gas_source=gas_source.replace('for seed in [42,7,101]:','for seed in [42]:').replace('for c in [.1,1.,10.]','for c in [1.]').replace("fit_conditional(d,y,'thermal',[42,7,101])","fit_conditional(d,y,'thermal',[42])").replace('for seeds in [[42,7,101],[11,23,37],[13,29,53]]:','for seeds in [[42]]:').replace('(vote>=2)','(vote>=1)')
exec(gas_source[:gas_source.index('root=Path.cwd()')])
def transform(imp,sc,x):
 return np.clip(sc.transform(imp.transform(x)),-12,12)
def fit_scale(x,normal):
 imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(x[normal]);sc=RobustScaler().fit(imp.transform(x)[normal])
 return imp,sc,transform(imp,sc,x)
def joint_features(x):
 j=np.maximum(0,np.column_stack([np.minimum(-x[:,0],x[:,2]),np.minimum(x[:,3],-x[:,4]),np.minimum(-x[:,5],x[:,6]),np.minimum(-x[:,0],-x[:,1]),np.minimum(x[:,7],-x[:,8]),np.minimum(x[:,9],x[:,10])]))
 return np.column_stack([j.max(axis=1),np.sort(j,axis=1)[:,-2],j.mean(axis=1),(j>1).sum(axis=1)])
def fit_mean_models(a,y,seeds,thermal):
 models=[]
 for cls in [RandomForestClassifier,ExtraTreesClassifier]:
  for seed in seeds:
   models.append((1/(5*len(seeds)),cls(n_estimators=100 if thermal else 120,max_depth=4,min_samples_leaf=2 if thermal else 3,class_weight='balanced',random_state=seed,n_jobs=2).fit(a,y)))
 for c in [.1,1.]:models.append((.2,SVC(C=c,class_weight='balanced',gamma='scale').fit(a,y)))
 models.append((.2,LogisticRegression(C=.1,class_weight='balanced',max_iter=3000).fit(a,y)))
 return models
def mean_scores(models,a):
 return sum(w*(expit(m.decision_function(a)) if isinstance(m,SVC) else m.predict_proba(a)[:,1]) for w,m in models)
def local_short(peer,x,context,timestamps):
 xx=peer['imp'].transform(x);cx=peer['sc'].transform(peer['ci'].transform(context))
 near=peer['nn'].kneighbors(cx,return_distance=False);out=[]
 for row,ids,ts in zip(xx,near,timestamps):
  ids=ids[peer['timestamps'][ids]!=np.datetime64(ts)][:30]
  if not len(ids):ids=np.arange(len(peer['x']))
  out.append((row-np.median(peer['x'][ids],axis=0))/np.maximum(peer['x'][ids].std(axis=0),peer['floor']))
 return np.asarray(out)
def predict_short(bundle,f):
 a=transform(bundle['A']['imp'],bundle['A']['sc'],f[bundle['columns']['std']].to_numpy(float))
 full=expit(bundle['A']['full'].decision_function(a))
 experts=np.max([mean_scores(ms,a) for ms in bundle['A']['modes']],axis=0)
 A=np.sqrt(full*experts)
 b=local_short(bundle['B']['peer'],f[bundle['columns']['physical']].to_numpy(float),f[bundle['columns']['context']].to_numpy(float),pd.to_datetime(f.TimeStamp).to_numpy())
 b=np.clip(bundle['B']['sc'].transform(b),-12,12)
 B=mean_scores(bundle['B']['models'],b)
 C=np.mean([mean_scores(c['models'],transform(c['imp'],c['sc'],f[bundle['columns']['shock']].to_numpy(float))) for c in bundle['C']],axis=0)
 dd=joint_features(transform(bundle['D']['imp'],bundle['D']['sc'],f[bundle['columns']['joint']].to_numpy(float)))
 D=bundle['D']['model'].predict_proba(dd)[:,1]
 cuts=bundle['cuts']
 flags=np.column_stack([A>=cuts['A'],B>=cuts['B'],(C>=cuts['C'])&(f.Barrel_Temperature_5__past10_std.to_numpy()>=cuts['barrel5']), (D>=cuts['D'])&(-f.Hopper_Temperature__dev10.to_numpy()>=cuts['hopper'])])
 return flags.any(axis=1),np.column_stack([A,B,C,D])


RAW_TEMP=[f'Barrel_Temperature_{i}' for i in range(1,7)]+['Hopper_Temperature','Mold_Temperature_3','Mold_Temperature_4']
def conditioned_routed(peer,joint,context,timestamps):
 query=peer['ri'].transform(joint)
 near=peer['nn'].kneighbors(peer['sc'].transform(peer['ci'].transform(context)),return_distance=False)
 result=[]
 for row,ids,t in zip(query,near,timestamps):
  ids=ids[peer['timestamps'][ids]!=np.datetime64(t)][:100]
  center=np.median(peer['raw'][ids],axis=0)
  spread=np.maximum(1.4826*np.median(abs(peer['raw'][ids]-center),axis=0),peer['floor'])
  result.append(np.clip((row-center)/spread,-12,12))
 return np.asarray(result)
def predict_routed_short(bundle,f):
 needed=set(RAW_TEMP+['TimeStamp','Barrel_Temperature_5__past10_std','Hopper_Temperature__dev10']+[c for cols in bundle['original15']['columns'].values() for c in cols])
 if needed-set(f):raise ValueError('Missing preprocessed shortshot columns: '+str(sorted(needed-set(f))))
 if bundle.get('format')!='RG3_SHORTSHOT_ROUTED_D_V1':raise ValueError('Wrong shortshot model format')
 old=bundle['original15']
 _,scores=predict_short(old,f);cuts=old['cuts']
 abc=(scores[:,0]>=cuts['A'])|(scores[:,1]>=cuts['B'])|((scores[:,2]>=cuts['C'])&(f.Barrel_Temperature_5__past10_std.to_numpy()>=cuts['barrel5']))
 oldD=(scores[:,3]>=cuts['D'])&(-f.Hopper_Temperature__dev10.to_numpy()>=cuts['hopper'])
 p=bundle['support'];context=f[RAW_TEMP].to_numpy(float)
 distance=p['nn'].kneighbors((p['imputer'].transform(context)-p['median'])/p['scale'],n_neighbors=1)[0][:,0]
 supported=distance<=p['cut']
 z=conditioned_routed(bundle['conditioned_peer'],f[old['columns']['joint']].to_numpy(float),context,pd.to_datetime(f.TimeStamp).to_numpy())
 newscore=bundle['D_model'].predict_proba(joint_features(z))[:,1]
 newD=(newscore>=bundle['D_cut'])&(-f.Hopper_Temperature__dev10.to_numpy()>=cuts['hopper'])
 routedD=np.where(supported,oldD,newD)
 return abc|routedD,{'old_state_supported':supported.astype(int),'D_routed':routedD.astype(int),'D_new_score':newscore}



In [ ]:
import shutil,gzip
rawdir=WORK/'data/raw_input';rawdir.mkdir(parents=True,exist_ok=True)
for name in ['labeled_data.csv','moldset_labeled.csv']:
 shutil.copy2(ROOT/'dataset/original'/name,rawdir/name)
with gzip.open(ROOT/'dataset/original/unlabeled_data.csv.gz','rb') as src,(rawdir/'unlabeled_data.csv').open('wb') as dst:shutil.copyfileobj(src,dst)
for p in (ROOT/'dataset/kamp').glob('*.csv'):shutil.copy2(p,rawdir/p.name)
legacy_original=WORK/'data/original';legacy_original.mkdir(parents=True,exist_ok=True)
for p in rawdir.glob('*.csv'):shutil.copy2(p,legacy_original/p.name)
for p in (ROOT/'dataset/kamp').glob('*.csv'):shutil.copy2(p,WORK/'data'/p.name)
# 기존 전처리를 새 실행 루트에서 재구성. 모델/예측 CSV를 입력으로 쓰지 않는다.
previous=Path.cwd()
try:
 os.chdir(WORK)
 prep=source('158_rg3_prepare_from_local_raw.ipynb')
 # 새 helper .py를 작성하지 않고 이미 제공된 함수와 노트북 사용.
 exec(prep,{'__name__':'__main__'})
 # 공식 가스12개 OOF를 처음부터 재현해 경계값 산정.
 n=nbformat.read(WORK/'notebooks/193_rg3_fresh12_full_reproduction.ipynb',4)
 ns={'__name__':'__main__'}
 for c in n.cells:
  if c.cell_type=='code':exec(c.source,ns)
 # 최고 미성형의 구간 제외 개발평가 재현.
 for filename in ['133_rg3_shortshot_sparse_modes.ipynb','139_rg3_shortshot_thermal_seed_majority.ipynb','141_rg3_shortshot_joint_tail_confirmation.ipynb','143_rg3_shortshot_fixed_candidate_review.ipynb']:
  if not (WORK/'notebooks'/filename).exists():
   matching=list((WORK/'notebooks').glob(filename.split('_')[0]+'_*'));assert len(matching)==1;filename=matching[0].name
  print('Shortshot reproduction',filename,flush=True)
  temp={'__name__':'__main__'}
  for c in nbformat.read(WORK/'notebooks'/filename,4).cells:
   if c.cell_type=='code':exec(c.source,temp)
 # 미성형145의 학습 정의만 사용; 비라벨 대입/예측은 실행하지 않음.
 setup=source('145_rg3_shortshot_unlabeled_application.ipynb')
 shortns={'__name__':'__main__'};exec(setup[:setup.index('u_all=pd.read_csv')],shortns)
finally:os.chdir(previous)
d=pd.read_csv(WORK/'reports/tables/original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
gas=d[d['product'].eq('RG3')&(d.PassOrFail.eq(0)|d.Reason.eq('가스'))].reset_index(drop=True)
cal=pd.read_csv(WORK/'reports/tables/rg3_193_fresh12_predictions.csv',parse_dates=['TimeStamp']).set_index('TimeStamp').loc[gas.TimeStamp]
normal=gas.PassOrFail.eq(0).to_numpy()
cuts={k:float(np.quantile(cal.loc[normal,k+'_score'],q,method='higher')) for k,q in [('baseline',.99),('expert',.995),('thermal',.975)]}
# 압력 CSV column 이름은 pressure_score.
cuts['pressure_42_7_101']=float(np.quantile(cal.loc[normal,'pressure_score'],.99,method='higher'))
dd=gas.Barrel_Temperature_5__delta1.to_numpy();cuts['barrel5_delta']=float(np.quantile(dd[normal&np.isfinite(dd)],.95,method='higher'))
zcal=[]
for block in gas.comparison_block.unique():
 q=np.flatnonzero(gas.comparison_block.eq(block));ref=np.flatnonzero(~gas.comparison_block.eq(block))
 zcal.extend(zip(q,ns['local_residual'](ref,q,'thermal')[:,4]))
cuts['thermal_score']=cuts.pop('thermal');cuts['thermal_gate']=float(np.quantile(np.array([v for _,v in sorted(zcal)])[normal],.05))
bundle=fit_gas_bundle(gas,cuts)
assert len(bundle['pressure'])==1 and sum(len(g) for g in bundle['thermal']['trees'])+len(bundle['thermal']['linear'])==3
save_bundle(bundle,MODEL_DIR/'gas12.joblib')

# Generate the confirmed15-classifier base and re-run state correction without using stale model predictions for training.
def run_internal(name):
 print('Internal rebuild',name,flush=True)
 execution={'__name__':'__main__'}
 previous=Path.cwd()
 try:
  os.chdir(WORK if name.startswith('204_') else ROOT)
  for cell in nbformat.read(WORK/'notebooks'/name,4).cells:
   if cell.cell_type=='code':exec(cell.source,execution)
 finally:os.chdir(previous)
 return execution
run_internal('204_rg3_shortshot15_actual_pruned_replay.ipynb')
base15=run_internal('205_rg3_shortshot15_saved_candidate.ipynb')
run_internal('207_rg3_shortshot_alarm_history_audit.ipynb')
run_internal('208_rg3_shortshot_D_alarm_correction.ipynb')
run_internal('209_rg3_shortshot_state_conditioned_D.ipynb')
pr={'__name__':'__main__'};previous=Path.cwd()
try:
 os.chdir(WORK);s=source('10_original_defect_report_revision.ipynb');exec(s.split('shots = label_shots.merge')[0],pr)
finally:os.chdir(previous)
gu=pr['observed'];gu=gu[gu.EQUIP_CD.eq('S14')&gu['product'].eq('RG3')]
go=predict_gas_features(bundle,gu);go['gas_candidate']=go.gas_prediction.eq(1).astype(int)
go.to_csv(WORK/'reports/tables/rg3_194_fresh12_unlabeled_predictions.csv',index=False)
run_internal('210_rg3_shortshot_D_state_routing.ipynb')
routed=run_internal('211_rg3_shortshot_routed_D_saved_checks.ipynb')
shutil.copy2(WORK/'models/shortshot_routed_D_candidate/shortshot_routed_D.joblib',MODEL_DIR/'shortshot_routed_D.joblib')
shutil.copy2(WORK/'models/shortshot_routed_D_candidate/verification.json',MODEL_DIR/'shortshot_routed_D.json')
import sklearn
metadata_path=MODEL_DIR/'shortshot_routed_D.json';metadata=json.loads(metadata_path.read_text(encoding='utf-8'))
metadata.update(sklearn_version=sklearn.__version__,numpy_version=np.__version__,confirmed_configuration='gas12 + shortshot16',new_state_shortshot_quality_validated=False,deployment_replaced=True,pushed=False)
metadata_path.write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
processed=ROOT/'dataset/processed/rg3';processed.mkdir(parents=True,exist_ok=True)
# Preserve the label-shot feature schema. Inference drops quality labels.
f=base15['f']
f.to_csv(processed/'rg3_input_features.csv',index=False,encoding='utf-8-sig')
base15['gas'].to_csv(processed/'rg3_gas_training.csv',index=False,encoding='utf-8-sig')
f[base15['use']].to_csv(processed/'rg3_shortshot_training.csv',index=False,encoding='utf-8-sig')
# Export same6590 historical unknown shots for direct saved-model inference, joining the independently rebuilt gas features by keys.
pr={'__name__':'__main__'};previous=Path.cwd()
try:
 os.chdir(WORK);s=source('10_original_defect_report_revision.ipynb');exec(s.split('shots = label_shots.merge')[0],pr)
finally:os.chdir(previous)
legacy=routed['f'];gas_unknown=pr['observed']
gas_unknown=gas_unknown[gas_unknown.EQUIP_CD.eq('S14')&gas_unknown['product'].eq('RG3')]
uf=legacy.merge(gas_unknown.drop(columns=['Barrel_Temperature_5__past10_std','Hopper_Temperature__dev10']),on='TimeStamp',validate='one_to_one')
assert len(uf)==6590
uf.to_csv(processed/'rg3_unlabeled_input_features.csv',index=False,encoding='utf-8-sig')
routed['ns']['pool_times'].shape
ref=base15['f'].TimeStamp.is_unique
assert ref
print('Confirmed deployment: gas12 + routed shortshot16 =28 classifiers')

